# PMLB Hybrid QNN Classification

This notebook loads a real PMLB classification dataset, trains a plain NN baseline and a hybrid QNN with linear concatenation, and saves the resulting metrics and predictions.

In [1]:
import copy
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from pmlb import fetch_data
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
PMLB_DATASET = "breast_cancer"
RESULTS_DIR = Path("/home/sammarv/quantum_corrosion/results/pmlb_hybrid_qnn_classification")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Dataset:", PMLB_DATASET)

Device: cpu
Dataset: breast_cancer


## 1. Set Up Imports and Configuration

Load the real PMLB dataset, prepare the training configuration, and create the results directory.

In [2]:
def load_pmlb_dataset(dataset_name: str) -> pd.DataFrame:
    data = fetch_data(dataset_name, dropna=True)
    if not isinstance(data, pd.DataFrame):
        raise TypeError(f'Expected a DataFrame from PMLB, got {type(data)!r}')
    return data


def split_and_scale_dataframe(data: pd.DataFrame, seed: int = SEED):
    if 'target' in data.columns:
        feature_df = data.drop(columns=['target'])
        target_series = data['target']
    else:
        feature_df = data.iloc[:, :-1]
        target_series = data.iloc[:, -1]

    if feature_df.empty:
        raise ValueError('PMLB dataset has no features')

    label_encoder = LabelEncoder()
    y_all = label_encoder.fit_transform(target_series.astype(str))
    X_all = feature_df.to_numpy(dtype=np.float32)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X_all, y_all, test_size=0.30, random_state=seed, stratify=y_all
    )
    X_valid, X_test, y_valid, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_valid = scaler.transform(X_valid).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    return {
        'data': data,
        'feature_names': feature_df.columns.to_list(),
        'label_encoder': label_encoder,
        'X_train': X_train,
        'X_valid': X_valid,
        'X_test': X_test,
        'y_train': y_train,
        'y_valid': y_valid,
        'y_test': y_test,
        'scaler': scaler,
    }


raw_df = load_pmlb_dataset(PMLB_DATASET)
prep = split_and_scale_dataframe(raw_df)
N_FEATURES = prep['X_train'].shape[1]
N_CLASSES = len(prep['label_encoder'].classes_)
CLASS_NAMES = prep['label_encoder'].classes_.tolist()

print('Raw shape:', raw_df.shape)
print('Feature count:', N_FEATURES)
print('Classes:', CLASS_NAMES)
print('Train/Valid/Test:', prep['X_train'].shape, prep['X_valid'].shape, prep['X_test'].shape)
print('Train class balance:', np.bincount(prep['y_train']))

Raw shape: (286, 10)
Feature count: 9
Classes: ['0', '1']
Train/Valid/Test: (200, 9) (43, 9) (43, 9)
Train class balance: [141  59]


## 2. Define Core Data Structures

Create the split tensors, loaders, and experiment configuration used by both the baseline and the hybrid model.

In [3]:
EXPERIMENT = {
    'dataset_name': PMLB_DATASET,
    'seed': SEED,
    'batch_size': 32,
    'epochs': 20,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'n_qubits': 4,
    'n_q_layers': 2,
    'train_split': 0.70,
    'valid_split': 0.15,
    'test_split': 0.15,
}


def to_tensor_dataset(X: np.ndarray, y: np.ndarray) -> TensorDataset:
    return TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )


train_ds = to_tensor_dataset(prep['X_train'], prep['y_train'])
valid_ds = to_tensor_dataset(prep['X_valid'], prep['y_valid'])
test_ds = to_tensor_dataset(prep['X_test'], prep['y_test'])

train_loader = DataLoader(train_ds, batch_size=EXPERIMENT['batch_size'], shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=EXPERIMENT['batch_size'], shuffle=False)
test_loader = DataLoader(test_ds, batch_size=EXPERIMENT['batch_size'], shuffle=False)

assert len(train_ds) > 0 and len(valid_ds) > 0 and len(test_ds) > 0
assert prep['X_train'].shape[1] == prep['X_valid'].shape[1] == prep['X_test'].shape[1]
print('Loaders:', len(train_loader), len(valid_loader), len(test_loader))

Loaders: 7 2 2


## 3. Implement Primary Functions

Define the plain neural network baseline, the compact quantum layer, and the hybrid QNN with linear concatenation fusion.

In [4]:
n_features = prep['X_train'].shape[1]
n_classes = len(CLASS_NAMES)
n_qubits = min(EXPERIMENT['n_qubits'], n_features)


def make_quantum_device(n_q):
    return qml.device('default.qubit', wires=n_q)


qdev = make_quantum_device(n_qubits)


@qml.qnode(qdev, interface='torch', diff_method='backprop')
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


class BaselineNN(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, output_dim),
        )

    def forward(self, x):
        return self.net(x)


class QuantumProjection(nn.Module):
    def __init__(self, input_dim: int, q_dim: int, q_layers: int):
        super().__init__()
        self.projector = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, q_dim),
            nn.Sigmoid(),
        )
        self.weights = nn.Parameter(torch.randn(q_layers, q_dim) * 0.1)
        self.q_dim = q_dim

    def forward(self, x):
        q_inputs = self.projector(x) * math.pi
        quantum_outputs = []
        for sample in q_inputs:
            q_out = qnode(sample, self.weights)
            if isinstance(q_out, list):
                q_out = torch.stack([
                    value if isinstance(value, torch.Tensor) else torch.as_tensor(value, dtype=sample.dtype, device=sample.device)
                    for value in q_out
                ])
            if not isinstance(q_out, torch.Tensor):
                q_out = torch.as_tensor(q_out, dtype=sample.dtype, device=sample.device)
            quantum_outputs.append(q_out.float())
        return torch.stack(quantum_outputs, dim=0)


class HybridQNNLinearConcat(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, q_dim: int, q_layers: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.quantum = QuantumProjection(64, q_dim, q_layers)
        self.head = nn.Linear(64 + q_dim, output_dim)

    def forward(self, x):
        c_feat = self.encoder(x)
        q_feat = self.quantum(c_feat)
        fused = torch.cat([c_feat, q_feat], dim=1)
        return self.head(fused)


def make_classification_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'classification_report': classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
    }


baseline_model = BaselineNN(n_features, n_classes).to(device)
hybrid_model = HybridQNNLinearConcat(n_features, n_classes, n_qubits, EXPERIMENT['n_q_layers']).to(device)
print('Baseline params:', sum(p.numel() for p in baseline_model.parameters() if p.requires_grad))
print('Hybrid params:', sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad))

Baseline params: 2786
Hybrid params: 14102


## 4. Handle Validation and Errors

Check the inputs, the split sizes, and a single forward pass from both models before starting full training.

In [5]:
def validate_batch(batch_x, batch_y):
    if batch_x.ndim != 2:
        raise ValueError(f'Expected 2D batch tensor, got shape {tuple(batch_x.shape)}')
    if batch_x.shape[1] != n_features:
        raise ValueError(f'Expected {n_features} features, got {batch_x.shape[1]}')
    if batch_y.ndim != 1:
        raise ValueError(f'Expected 1D label tensor, got shape {tuple(batch_y.shape)}')


sample_x, sample_y = next(iter(train_loader))
validate_batch(sample_x, sample_y)

with torch.no_grad():
    baseline_logits = baseline_model(sample_x.to(device))
    hybrid_logits = hybrid_model(sample_x.to(device))

assert baseline_logits.shape == (sample_x.shape[0], n_classes)
assert hybrid_logits.shape == (sample_x.shape[0], n_classes)
print('Validation passed: baseline and hybrid forward shapes are correct.')

Validation passed: baseline and hybrid forward shapes are correct.


## 5. Run a Minimal Usage Example

Train the baseline and hybrid models on the full real PMLB split and keep the best validation checkpoint for each model.

In [6]:
def evaluate_model(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    y_prob = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            y_true.extend(yb.numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
            y_prob.extend(probs.cpu().numpy().tolist())
    return np.array(y_true), np.array(y_pred), np.array(y_prob)


def train_model(model, train_loader, valid_loader, epochs=15, lr=1e-3, weight_decay=1e-4, patience=5):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    best_state = copy.deepcopy(model.state_dict())
    best_score = -1.0
    best_epoch = -1
    stale = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_true = []
        train_pred = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
            train_true.extend(yb.cpu().numpy().tolist())
            train_pred.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

        train_loss /= len(train_loader.dataset)
        train_acc = accuracy_score(train_true, train_pred)
        val_true, val_pred, _ = evaluate_model(model, valid_loader)
        val_metrics = make_classification_metrics(val_true, val_pred)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc, **val_metrics})
        print(f'Epoch {epoch:02d}/{epochs} train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_acc={val_metrics["accuracy"]:.4f} val_f1={val_metrics["macro_f1"]:.4f}')

        if val_metrics['macro_f1'] > best_score:
            best_score = val_metrics['macro_f1']
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    return model, history, {'best_epoch': best_epoch, 'best_macro_f1': best_score}


baseline_model, baseline_history, baseline_best = train_model(
    baseline_model,
    train_loader,
    valid_loader,
    epochs=EXPERIMENT['epochs'],
    lr=EXPERIMENT['learning_rate'],
    weight_decay=EXPERIMENT['weight_decay'],
)

hybrid_model, hybrid_history, hybrid_best = train_model(
    hybrid_model,
    train_loader,
    valid_loader,
    epochs=EXPERIMENT['epochs'],
    lr=EXPERIMENT['learning_rate'],
    weight_decay=EXPERIMENT['weight_decay'],
)

print('Baseline best:', baseline_best)
print('Hybrid best:', hybrid_best)

Epoch 01/20 train_loss=0.6677 train_acc=0.6900 val_acc=0.6744 val_f1=0.4028
Epoch 02/20 train_loss=0.6327 train_acc=0.6950 val_acc=0.6977 val_f1=0.4110
Epoch 03/20 train_loss=0.6070 train_acc=0.7050 val_acc=0.6977 val_f1=0.4110
Epoch 04/20 train_loss=0.5887 train_acc=0.7100 val_acc=0.6977 val_f1=0.4110
Epoch 05/20 train_loss=0.5658 train_acc=0.7050 val_acc=0.6977 val_f1=0.4110
Epoch 06/20 train_loss=0.5588 train_acc=0.7050 val_acc=0.7209 val_f1=0.4881
Epoch 07/20 train_loss=0.5538 train_acc=0.7250 val_acc=0.7209 val_f1=0.4881
Epoch 08/20 train_loss=0.5388 train_acc=0.7350 val_acc=0.7209 val_f1=0.4881
Epoch 09/20 train_loss=0.5328 train_acc=0.7500 val_acc=0.7209 val_f1=0.5393
Epoch 10/20 train_loss=0.5263 train_acc=0.7550 val_acc=0.7442 val_f1=0.5968
Epoch 11/20 train_loss=0.5204 train_acc=0.7750 val_acc=0.7674 val_f1=0.6487
Epoch 12/20 train_loss=0.5262 train_acc=0.7450 val_acc=0.7907 val_f1=0.6960
Epoch 13/20 train_loss=0.5151 train_acc=0.7700 val_acc=0.7907 val_f1=0.6960
Epoch 14/20 

## 6. Add Basic Tests

Evaluate both trained models on the held-out test set, assert the outputs are finite, and save the final artifacts.

In [7]:
baseline_test_true, baseline_test_pred, baseline_test_prob = evaluate_model(baseline_model, test_loader)
hybrid_test_true, hybrid_test_pred, hybrid_test_prob = evaluate_model(hybrid_model, test_loader)

baseline_metrics = make_classification_metrics(baseline_test_true, baseline_test_pred)
hybrid_metrics = make_classification_metrics(hybrid_test_true, hybrid_test_pred)

assert np.isfinite(baseline_test_prob).all()
assert np.isfinite(hybrid_test_prob).all()
assert len(baseline_test_true) == len(baseline_test_pred)
assert len(hybrid_test_true) == len(hybrid_test_pred)

print('\nBaseline classification report')
print(baseline_metrics['classification_report'])
print('\nHybrid classification report')
print(hybrid_metrics['classification_report'])

baseline_predictions = pd.DataFrame({
    'y_true': baseline_test_true,
    'y_pred': baseline_test_pred,
})
hybrid_predictions = pd.DataFrame({
    'y_true': hybrid_test_true,
    'y_pred': hybrid_test_pred,
})

baseline_model_path = RESULTS_DIR / 'baseline_nn.pt'
hybrid_model_path = RESULTS_DIR / 'hybrid_qnn_linear_concat.pt'
baseline_predictions_path = RESULTS_DIR / 'baseline_predictions.csv'
hybrid_predictions_path = RESULTS_DIR / 'hybrid_predictions.csv'
metrics_path = RESULTS_DIR / 'metrics.json'

torch.save(baseline_model.state_dict(), baseline_model_path)
torch.save(hybrid_model.state_dict(), hybrid_model_path)
baseline_predictions.to_csv(baseline_predictions_path, index=False)
hybrid_predictions.to_csv(hybrid_predictions_path, index=False)

metrics_payload = {
    'config': EXPERIMENT,
    'dataset_name': PMLB_DATASET,
    'n_features': n_features,
    'n_classes': n_classes,
    'class_names': CLASS_NAMES,
    'baseline_best': baseline_best,
    'hybrid_best': hybrid_best,
    'baseline_metrics': baseline_metrics,
    'hybrid_metrics': hybrid_metrics,
}
with open(metrics_path, 'w') as f:
    json.dump(metrics_payload, f, indent=2)

print('Saved baseline model:', baseline_model_path)
print('Saved hybrid model:', hybrid_model_path)
print('Saved baseline predictions:', baseline_predictions_path)
print('Saved hybrid predictions:', hybrid_predictions_path)
print('Saved metrics:', metrics_path)


Baseline classification report
              precision    recall  f1-score   support

           0       0.73      0.90      0.81        30
           1       0.50      0.23      0.32        13

    accuracy                           0.70        43
   macro avg       0.61      0.57      0.56        43
weighted avg       0.66      0.70      0.66        43


Hybrid classification report
              precision    recall  f1-score   support

           0       0.73      0.90      0.81        30
           1       0.50      0.23      0.32        13

    accuracy                           0.70        43
   macro avg       0.61      0.57      0.56        43
weighted avg       0.66      0.70      0.66        43

Saved baseline model: /home/sammarv/quantum_corrosion/results/pmlb_hybrid_qnn_classification/baseline_nn.pt
Saved hybrid model: /home/sammarv/quantum_corrosion/results/pmlb_hybrid_qnn_classification/hybrid_qnn_linear_concat.pt
Saved baseline predictions: /home/sammarv/quantum_corrosi